In [1]:
"""B64 light-current three-point analysis and paper plots."""

from pathlib import Path
import os
import sys

import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

HERE = Path.cwd().resolve()
if HERE.parent.name == "__codex_ignore":
    HERE = HERE.parent.parent / HERE.name
if HERE.name != "cB211.072.64" or HERE.parent.name != "07_Nsgm":
    raise RuntimeError("Launch this notebook from its cB211.072.64 directory.")
WORK = HERE.parent / "__codex_ignore" / HERE.name
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(HERE.parent))

import util as yu
import util_codex as yuc

yu.setpath("analysis_3pt_light_codex")
ENS = "b"
TFS = [8, 10, 12, 14, 16, 18, 20]
LOWER_BOUNDS = [8, 10, 12, 14, 16, 18]
XUNIT = yu.ens2a[ENS]
YUNIT = yu.ens2amul_iso[ENS] * yu.ens2aInv[ENS]
AINV_GEV = yu.ens2aInv[ENS] / 1000
SINGLE_DELTA = 2
STANDARD_CUT = 2
LAPLACE_CUT = 4


# Analysis helpers
Laplace transformations stay local because they are specific to this analysis.


In [2]:
def laplace_c3(tf2c3pt, energy, displacement):
    energy = np.asarray(energy)
    eigenvalue = 2 * np.cosh(displacement * energy) - 2
    c3_eigenvalue = eigenvalue if energy.ndim == 0 else eigenvalue[:, None]
    filtered = {}
    for tf, c3pt in tf2c3pt.items():
        second_difference = np.roll(c3pt, -displacement, 1) + np.roll(c3pt, displacement, 1) - 2 * c3pt
        filtered[tf] = -second_difference + c3_eigenvalue * c3pt
    return filtered, eigenvalue


In [3]:
def laplace_ratio(tf2c3pt, tf2c2pt, energy, displacement):
    filtered, eigenvalue = laplace_c3(tf2c3pt, energy, displacement)
    return {tf: c3pt / (eigenvalue * tf2c2pt[tf])[:, None] for tf, c3pt in filtered.items()}


In [4]:
def fit_single_laplace(displacement):
    cut = displacement + 2
    fits = yu.doFits_3pt_lbd(
        lambda energy: laplace_ratio(c3_standard, c2_standard, energy, displacement),
        LOWER_BOUNDS, [cut], symmetrizeQ=True,
        label=f"RLap_delta{displacement}_codex", overwrite=False,
    )
    return [[label, np.column_stack([pars[:, 0], np.abs(pars[:, 1])]), chi2, ndof]
            for label, pars, chi2, ndof in fits]


In [5]:
def fit_probability(fit):
    return yu.chi2Ndof2pval(np.mean(fit[2]), fit[3])


In [6]:
def select(fits, label):
    return next(fit for fit in fits if fit[0] == label)


In [7]:
def print_scan(name, fits, cut, energy=False):
    print(f"\n{name}")
    for fit in sorted((fit for fit in fits if fit[0][1] == cut), key=lambda fit: fit[0][0]):
        sigma = yu.jackme_un2str(fit[1][:, 0] * YUNIT)
        gap = f", gap={yu.jackme_un2str(fit[1][:, 1] * yu.ens2aInv[ENS])} MeV" if energy else ""
        print(f"  {fit[0]}: sigma={sigma} MeV{gap}, p={fit_probability(fit):.3f}")


# Analysis
Build the ratios first, fit the standard and single-filtered ratios, and form the double-filtered comparison without fitting it.


In [8]:
c2pt_matrix, tf2c3pt_matrix, c2pt_matched = yu.load_pkl(HERE / "pkl/processData/reg_ignore/data.pkl")
tf2c3pt_matrix = {tf: np.real(values) for tf, values in tf2c3pt_matrix.items()}
standard_2pt = yu.load_pkl_reg("standard_two_state_selected", pathlabel="analysis_2pt_codex")
nucleon_standard = standard_2pt["N"]
nsigma_selected = yu.load_pkl_reg("two_state_selected", pathlabel="analysis_2pt_codex")["Nsgm"]
nucleon_two_state = nucleon_standard
v, _, w = yu.load_pkl_reg("evec_ratios", pathlabel="analysis_2pt_codex")


## Standard and reduced-GEVP ratios


In [9]:
c3_standard = {tf: tf2c3pt_matrix[tf][:, :, 0, 0] for tf in TFS}
c2_standard = {tf: c2pt_matched[tf] for tf in TFS}
c3_gevp = {}
c2_gevp = {}
for tf in TFS:
    c3pt = tf2c3pt_matrix[tf]
    v_column, w_column = v[:, None], w[:, None]
    c3_gevp[tf] = (1 - w_column**2) * c3pt[:, :, 0, 0]
    c3_gevp[tf] += v_column * (1 + w_column) * (c3pt[:, :, 0, 1] + c3pt[:, :, 1, 0])
    c2_gevp[tf] = c2pt_matched[tf] + v * (c2pt_matrix[:, tf, 0, 1] + c2pt_matrix[:, tf, 1, 0])
    c2_gevp[tf] += v**2 * c2pt_matrix[:, tf, 1, 1]

ratio_standard = {tf: c3_standard[tf] / c2_standard[tf][:, None] for tf in TFS}
ratio_gevp = {tf: c3_gevp[tf] / c2_gevp[tf][:, None] for tf in TFS}

yuc.guard_fit_cache(nucleon_two_state, *ratio_standard.values(), *ratio_gevp.values())


## Standard-ratio fits


In [10]:
fits_standard_shared = yu.doFits_3pt(
    "2st2step_SYMshare", ratio_standard, LOWER_BOUNDS, [STANDARD_CUT],
    pars_jk_meff2st=nucleon_two_state, symmetrizeQ=True,
    label="Rstd_shared_codex", overwrite=False,
)
fits_standard_free = yu.doFits_3pt(
    "2st2step_SYM", ratio_standard, LOWER_BOUNDS, [STANDARD_CUT],
    pars_jk_meff2st=nucleon_two_state, symmetrizeQ=True,
    label="Rstd_free_codex", overwrite=False,
)
fits_standard_split_gap = yu.doFits_3pt(
    "2st2step_SYM_share11", ratio_standard, LOWER_BOUNDS, [STANDARD_CUT],
    pars_jk_meff2st=nucleon_two_state, symmetrizeQ=True,
    label="Rstd_split_gap_codex", overwrite=False,
)
fits_standard_no_diagonal = yu.doFits_3pt(
    "2st2step_SYM_0rc1_0ra11", ratio_standard, LOWER_BOUNDS, [STANDARD_CUT],
    symmetrizeQ=True, label="Rstd_no_diagonal_codex", overwrite=False,
)

fits_standard_zero_r11 = yu.doFits_3pt(
    "2st2step_SYM_0ra11", ratio_standard, LOWER_BOUNDS, [STANDARD_CUT],
    pars_jk_meff2st=nucleon_two_state, symmetrizeQ=True,
    label="Rstd_zero_r11_codex", overwrite=False,
)


## Laplace fits and filter gaps


In [11]:
single_laplace_fits = {displacement: fit_single_laplace(displacement) for displacement in [1, 2, 3]}
selected_laplace_fits = {
    displacement: select(fits, (10, displacement + 2))
    for displacement, fits in single_laplace_fits.items()
}
selected_standard = select(fits_standard_zero_r11, (12, STANDARD_CUT))
standard_gap = selected_standard[1][:, 1]
yu.save_pkl_reg("standard_ratio_selected", dict(case="IV", window=selected_standard[0],
    fit=selected_standard, gap=standard_gap))
print("Selected light IV:", yu.jackme_un2str(selected_standard[1][:, 0] * YUNIT),
      "MeV; gap:", yu.jackme_un2str(standard_gap * yu.ens2aInv[ENS]), "MeV")
print("Selected E1:", yu.jackme_un2str((nucleon_standard[:, 0] + standard_gap) * AINV_GEV), "GeV")
laplace_gap = selected_laplace_fits[SINGLE_DELTA][1][:, 1]
ratio_laplace = laplace_ratio(c3_standard, c2_standard, laplace_gap, SINGLE_DELTA)
ratio_laplace_gevp = laplace_ratio(c3_gevp, c2_gevp, laplace_gap, SINGLE_DELTA)


Selected light IV: 45.9(3.0) MeV; gap: 405(35) MeV
Selected E1: 1.349(34) GeV


## Fits to the filtered GEVP ratio


In [12]:
fits_laplace_gevp = yu.doFits_3pt(
    "2st2step_SYM_0rc1_0ra11", ratio_laplace_gevp, [8, 10, 12, 14, 16], [LAPLACE_CUT],
    symmetrizeQ=True, label="RLG_caseIV_codex", overwrite=False,
)
fits_laplace_gevp_iv = yu.doFits_3pt(
    "2st2step_SYM_0ra11", ratio_laplace_gevp, [8, 10, 12, 14, 16], [LAPLACE_CUT],
    pars_jk_meff2st=nucleon_two_state, symmetrizeQ=True,
    label="RLG_zero_r11_std2pt_codex_" + yuc.sample_cache_tag(
        nucleon_two_state, laplace_gap, *ratio_laplace_gevp.values()),
    overwrite=False, verbose=2,
)
selected_laplace_gevp = select(fits_laplace_gevp_iv, (10, LAPLACE_CUT))
rlg_gap = selected_laplace_gevp[1][:, 1]
laplace_response = lambda gap: 1 - (2 * np.cosh(SINGLE_DELTA * gap) - 2) / (2 * np.cosh(SINGLE_DELTA * laplace_gap) - 2)
rlg_response = laplace_response(rlg_gap)
nsigma_response = laplace_response(nsigma_selected[:, 1])
gevp_residual_overlap = nucleon_standard[:, 2] * nsigma_selected[:, 2]

first_delta, second_delta = 2, 3
first_gap = selected_laplace_fits[first_delta][1][:, 1]
first_c3, first_eigenvalue = laplace_c3(c3_gevp, first_gap, first_delta)
sigma_gap = nucleon_standard[:, 1] + first_gap
second_c3, second_eigenvalue = laplace_c3(first_c3, sigma_gap, second_delta)
ratio_double_laplace = {
    tf: values / (first_eigenvalue * second_eigenvalue * c2_gevp[tf])[:, None]
    for tf, values in second_c3.items()
}
double_cut = first_delta + second_delta + 1

print_scan("standard shared gap", fits_standard_shared, STANDARD_CUT)
print_scan("standard free gap", fits_standard_free, STANDARD_CUT, energy=True)
print_scan("standard split gap", fits_standard_split_gap, STANDARD_CUT, energy=True)
print_scan("standard no diagonal", fits_standard_no_diagonal, STANDARD_CUT, energy=True)
print_scan("single Laplace", single_laplace_fits[SINGLE_DELTA], LAPLACE_CUT, energy=True)
print_scan("Laplace on GEVP, case IV", fits_laplace_gevp_iv, LAPLACE_CUT, energy=True)
print_scan("Laplace on GEVP, case V", fits_laplace_gevp, LAPLACE_CUT, energy=True)
print("normalized filter response for RLG gap:", yu.jackme_un2str(rlg_response))
print("normalized filter response for Nsigma excited gap:", yu.jackme_un2str(nsigma_response))
print("idealized residual GEVP overlap ratio:", yu.jackme_un2str(gevp_residual_overlap))



standard shared gap
  (8, 2): sigma=38.5(2.7) MeV, p=0.082
  (10, 2): sigma=39.6(2.5) MeV, p=0.251
  (12, 2): sigma=39.7(2.4) MeV, p=0.164
  (14, 2): sigma=40.0(2.4) MeV, p=0.105
  (16, 2): sigma=43.3(2.8) MeV, p=0.427
  (18, 2): sigma=45.9(3.1) MeV, p=0.505

standard free gap
  (8, 2): sigma=46.4(3.3) MeV, gap=393(39) MeV, p=0.374
  (10, 2): sigma=45.5(3.1) MeV, gap=416(43) MeV, p=0.485
  (12, 2): sigma=45.8(3.5) MeV, gap=407(48) MeV, p=0.341
  (14, 2): sigma=50.0(5.7) MeV, gap=350(58) MeV, p=0.391
  (16, 2): sigma=53.0(7.5) MeV, gap=359(78) MeV, p=0.721
  (18, 2): sigma=51.9(6.9) MeV, gap=403(94) MeV, p=0.611

standard split gap
  (8, 2): sigma=47.5(4.0) MeV, gap=383(41) MeV, p=0.401
  (10, 2): sigma=45.5(3.2) MeV, gap=414(44) MeV, p=0.483
  (12, 2): sigma=46.1(4.0) MeV, gap=402(52) MeV, p=0.341
  (14, 2): sigma=52.6(9.7) MeV, gap=328(76) MeV, p=0.415
  (16, 2): sigma=52.1(8.0) MeV, gap=356(86) MeV, p=0.705
  (18, 2): sigma=50.6(5.9) MeV, gap=403(94) MeV, p=0.609

standard no diagon

# Plotting helpers
The plotting layer consumes completed fit objects and does not perform analysis choices.


In [13]:
PLOT_STYLE = yuc.paper_style()

def ratio_dictionary(ratio, cut, open_symbol, shift=0):
    return {
        "tf2ratio": yu.symmetrizeRatio(ratio),
        "rainbow:[tfmin,tfmax,tcmin,dt]": [None, None, cut, None],
        "xyunit": (XUNIT, YUNIT),
        "mfc:[global]": ["white" if open_symbol else None],
        "shift:[rainbow,midpoint,fit]": [shift, 3 * shift, 0],
    }

PLOT_CONFIG = {
    "limits": dict(ylim=(0, 80), yticks=np.arange(0, 81, 20)),
    "rainbow": dict(xlim=(-.82, .82), xticks=np.arange(-.6, .61, .3)),
    "midpoint": dict(xlim=(.55, 1.72), xticks=[.75, 1, 1.25, 1.5]),
    "scan": dict(xlim=(.55, 1.55), xticks=[.75, 1, 1.25, 1.5]),
    "gap": dict(ylim=(.2, .7)), "selection": (12, STANDARD_CUT), "selection_case": "IV",
}

PLOT_CONFIG["fit_styles"] = [
    ("red", "o", "I", -.4, False), ("green", "^", "II", -.2, True),
    ("blue", "s", "III", 0, True), ("purple", "v", "IV", .2, True),
    ("darkorange", "d", "V", .4, True),
]


In [14]:
def start_ratio_plot(baseline, transformed, cuts, labels, columns=3):
    return yuc.start_ratio_plot(baseline, transformed, cuts, labels, columns, xunit=XUNIT, yunit=YUNIT, limits=PLOT_CONFIG["limits"])


In [15]:
def plot_scan(axis, fits, cut, color, marker, label, shift=0, energy=False, selection=None, open_symbol=False):
    return yuc.plot_scan(axis, fits, cut, color, marker, label, shift, energy, selection, open_symbol, xunit=XUNIT, yunit=YUNIT, energy_unit=AINV_GEV)


In [16]:
compact_four_panels = yuc.compact_four_panels


In [17]:
def draw_ratio_pair(axis, baseline, transformed, cuts, labels, midpoint_axis=None, tfmin=None, legend_loc="lower left"):
    return yuc.draw_ratio_pair(axis, baseline, transformed, cuts, labels, midpoint_axis, tfmin, legend_loc, xunit=XUNIT, yunit=YUNIT, limits=PLOT_CONFIG["limits"])


In [18]:
def plot_case_iii_matrix_element():
    scan = sorted((fit for fit in fits_standard_split_gap
                   if fit[0][1] == STANDARD_CUT and fit[0][0] <= 14),
                  key=lambda fit: fit[0][0])
    x = np.array([fit[0][0] for fit in scan]) * XUNIT
    samples = [fit[1][:, 3] / nucleon_standard[:, 2] * YUNIT for fit in scan]
    values = np.array([yu.jackme(data) for data in samples])

    figure, axis = plt.subplots(figsize=(3.4, 2.35))
    yu.errorbar(axis, x, values[:, 0], values[:, 1], color="blue", fmt="s")
    yu.addRefLine(axis, 0, color="grey", ls="--", lw=.8, zorder=0)
    axis.set(xlabel=r"$t_s^{\rm low}$ [fm]",
             ylabel=r"$\langle1|\mathcal{O}|1\rangle$ [MeV]",
             xlim=(.55, 1.20), xticks=[.6, .8, 1.0, 1.2])
    figure.tight_layout()
    yu.finalizePlot("caseIII_excited_matrix_element", tightQ=False)

    print("\ncase III: <1|O|1> [MeV]")
    for fit, value in zip(scan, values):
        print(f"t_s^low/a={fit[0][0]}: {yu.un2str(value[0], value[1])}")


In [19]:
def plot_standard_gevp():
    yuc.plot_standard_gevp(ratio_standard, ratio_gevp,
        [fits_standard_shared, fits_standard_free, fits_standard_split_gap, fits_standard_zero_r11, fits_standard_no_diagonal],
        nucleon_standard, nsigma_selected, XUNIT, YUNIT, AINV_GEV, PLOT_CONFIG, "Rstd_RGEVP")


In [20]:
def gevp_variant(w_enabled=True, diagonal_enabled=True):
    result = {}
    for tf, c3pt in tf2c3pt_matrix.items():
        v_column = v[:, None]
        w_column = w[:, None] if w_enabled else 0
        numerator = (1 - w_column**2) * c3pt[:, :, 0, 0]
        numerator += v_column * (1 + w_column) * (c3pt[:, :, 0, 1] + c3pt[:, :, 1, 0])
        denominator = c2pt_matched[tf] + v * (c2pt_matrix[:, tf, 0, 1] + c2pt_matrix[:, tf, 1, 0])
        if diagonal_enabled:
            denominator += v**2 * c2pt_matrix[:, tf, 1, 1]
        result[tf] = numerator / denominator[:, None]
    return result


In [21]:
def plot_gevp_variants():
    variants = [
        (gevp_variant(False), "o", "white", -.055, r"$R_{\rm GEVP}^{\prime}$"),
        (ratio_gevp, "o", None, 0, r"$R_{\rm GEVP}^{d}$"),
        (gevp_variant(False, False), "d", "white", .055, r"$R_{\rm GEVP}^{\prime\prime}$"),
    ]
    figure, axis = plt.subplots(figsize=(3.4, 2.35))
    for ratio, marker, face, shift, _ in variants:
        ratio = yu.symmetrizeRatio(ratio)
        for index, tf in enumerate(sorted(ratio)):
            times = np.arange(1, tf)
            times = times[np.abs(times - tf // 2) % 3 == 0]
            mean, error = yu.jackme(ratio[tf])
            x = (times - tf / 2 + .05 * (index - len(ratio) / 2)) * XUNIT + shift
            color = yu.colors16[index]
            yu.errorbar(axis, x, mean[times] * YUNIT, error[times] * YUNIT, color=color,
                        fmt=marker, mfc=color if face is None else face, markersize=3.4)
    axis.set(xlabel=r"$t_{\rm ins}-t_s/2$ [fm]", ylabel=r"$\sigma_{\pi N}$ [MeV]",
             ylim=(0, 80), yticks=np.arange(0, 81, 20),
             xlim=(-.82, .82), xticks=np.arange(-.6, .61, .3))
    handles = [mpl.lines.Line2D([], [], color="black", marker=marker, ls="",
                 mfc="black" if face is None else face, label=label)
               for _, marker, face, _, label in variants]
    axis.legend(handles=handles, loc="upper center", ncols=3, fontsize=7,
                columnspacing=.8, handletextpad=.3)
    figure.tight_layout()
    yu.finalizePlot("Rd_compare_w_v2", tightQ=False)


In [22]:
def plot_laplace_summary():
    yuc.plot_laplace_summary(ratio_standard, ratio_laplace, ratio_gevp, ratio_laplace_gevp,
        [fits_standard_no_diagonal, single_laplace_fits[SINGLE_DELTA], fits_laplace_gevp],
        XUNIT, YUNIT, AINV_GEV, PLOT_CONFIG, "Rstd_RLap",
        fits_laplace_gevp_iv=fits_laplace_gevp_iv)


In [23]:
def plot_double_laplace():
    config = {**PLOT_CONFIG, "midpoint": dict(xlim=(.55, 1.72), xticks=[.8, 1.2, 1.6])}
    yuc.plot_double_laplace(ratio_gevp, ratio_double_laplace, double_cut, XUNIT, YUNIT, config)


In [24]:
def plot_displacement_checks():
    figure, axes = plt.subplots(1, 2, figsize=(3.4, 1.9))
    displacements = np.arange(1, 4)
    for axis, column, unit, label, limits in [
        (axes[0], 0, YUNIT, r"$\sigma_{\pi N}$ [MeV]", (0, 80)),
        (axes[1], 1, AINV_GEV, r"$\Delta E_1^{\rm 3pt,Lap}$ [GeV]", (.2, .65)),
    ]:
        values = np.array([yu.jackme(selected_laplace_fits[d][1][:, column] * unit) for d in displacements])
        yuc.errorbar_with_selected(
            axis, displacements, values[:, 0], values[:, 1], 1,
            color="darkorange", fmt="s", mfc="darkorange",
        )
        axis.set(xlabel=r"$\delta/a$", ylabel=label, xlim=(.7, 3.3), xticks=displacements, ylim=limits)
    figure.tight_layout(w_pad=.5)
    yu.finalizePlot("RLap_delta_dependence", tightQ=False)



In [25]:
def plot_energy_scales():
    yuc.plot_energy_scales(ENS, nucleon_standard, nsigma_selected, standard_gap, laplace_gap,
                          rlg_gap, AINV_GEV, dict(xlim=(.85, 2.42), xticks=np.arange(1, 2.41, .2)))


# Plotting
Generate all light-current figures after the analysis objects above are fixed.


In [26]:
with mpl.rc_context(PLOT_STYLE):
    plot_standard_gevp()
    plot_case_iii_matrix_element()
    plot_gevp_variants()
    plot_laplace_summary()
    plot_double_laplace()
    plot_displacement_checks()
    plot_energy_scales()



case III: <1|O|1> [MeV]
t_s^low/a=8: 13(13)
t_s^low/a=10: -11(16)
t_s^low/a=12: 2(28)
t_s^low/a=14: 47(75)
